In [7]:
"""
U/I AutoRec Blend + LGB + XGB + CatBoost
====================================================================
  1. Blend weight alpha is optimised per fold (search over 0.0 to 1.0)
     instead of fixed 50/50. alpha=0 means pure Item-AutoRec.
  2. Features include both User and Item predictions separately so classifiers
     can learn their own weighting.
"""

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import warnings
warnings.filterwarnings("ignore")

DATA_DIR = "/kaggle/input/ml-100k"
COLS     = ["user", "item", "rating", "timestamp"]
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


class AutoRec(nn.Module):
    def __init__(self, n_inputs, hidden=512, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(n_inputs, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_inputs),
        )
    def forward(self, x):
        return self.net(x)


def train_autorec(R_tr, R_val, seed=42,
                  n_epochs=500, hidden=512, lr=1e-3,
                  batch_size=256, dropout=0.3, reg=1e-2, patience=30):
    torch.manual_seed(seed)
    n_vecs, n_inputs = R_tr.shape

    vec_means = np.where(
        (R_tr != 0).sum(1) > 0,
        R_tr.sum(1) / (R_tr != 0).sum(1).clip(1), 0.0)

    def centre(R):
        Rc = R.copy()
        for i in range(n_vecs):
            idx = Rc[i] != 0
            Rc[i, idx] -= vec_means[i]
        return Rc

    Rt = torch.FloatTensor(centre(R_tr )).to(DEVICE)
    Rv = torch.FloatTensor(centre(R_val)).to(DEVICE)
    mt = (Rt != 0).float()
    mv = (Rv != 0).float()

    model = AutoRec(n_inputs, hidden, dropout).to(DEVICE)
    opt   = optim.Adam(model.parameters(), lr=lr, weight_decay=reg)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs, eta_min=1e-5)
    loader = DataLoader(TensorDataset(Rt, mt), batch_size=batch_size, shuffle=True)

    best_val, best_state, wait = float("inf"), None, 0

    for epoch in range(1, n_epochs + 1):
        model.train()
        for xb, mb in loader:
            opt.zero_grad()
            loss = ((model(xb) - xb)**2 * mb).sum() / mb.sum()
            loss.backward()
            opt.step()
        sched.step()

        model.eval()
        with torch.no_grad():
            vl = ((model(Rv) - Rv)**2 * mv).sum() / mv.sum()
        vl = vl.item()

        if vl < best_val - 1e-5:
            best_val, wait = vl, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            wait += 1
            if wait >= patience:
                print(f"      early stop @ epoch {epoch}  val_loss={best_val:.4f}", flush=True)
                break
        if epoch % 100 == 0:
            print(f"      epoch {epoch:4d}  val_loss={vl:.4f}", flush=True)

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        R_hat_c = model(Rt).cpu().numpy()
    return np.clip(R_hat_c + vec_means[:, None], 1, 5)


def train_ensemble(R_tr, R_val, n_seeds=3, **kwargs):
    return np.mean([train_autorec(R_tr, R_val, seed=s, **kwargs)
                    for s in [42, 7, 99][:n_seeds]], axis=0)


def build_rating_matrix(train_df):
    users = sorted(train_df["user"].unique())
    items = sorted(train_df["item"].unique())
    u2i   = {u: i for i, u in enumerate(users)}
    i2i   = {it: i for i, it in enumerate(items)}
    R     = np.zeros((len(users), len(items)), dtype=np.float32)
    R[train_df["user"].map(u2i).values,
      train_df["item"].map(i2i).values] = train_df["rating"].values
    return R, u2i, i2i


def make_val_split(R_full, seed):
    R_tr = R_full.copy(); R_val = np.zeros_like(R_full)
    rng  = np.random.default_rng(seed)
    for u in range(R_full.shape[0]):
        rated = np.where(R_full[u] != 0)[0]
        if len(rated) >= 5:
            val_idx = rng.choice(rated, max(1, int(0.1*len(rated))), replace=False)
            R_val[u, val_idx] = R_full[u, val_idx]
            R_tr[u,  val_idx] = 0.0
    return R_tr, R_val


def get_preds(df, R_hat_u, R_hat_i, u2i, i2i, gm, user_mean_vec, item_mean_vec):
    uid  = df["user"].map(u2i).fillna(-1).astype(int).values
    iid  = df["item"].map(i2i).fillna(-1).astype(int).values
    both = (uid >= 0) & (iid >= 0)

    u_pred = np.full(len(df), gm, dtype=np.float32)
    i_pred = np.full(len(df), gm, dtype=np.float32)

    if both.any():
        u_pred[both] = R_hat_u[uid[both], iid[both]]
        i_pred[both] = R_hat_i[uid[both], iid[both]]

    uo = (uid >= 0) & (iid < 0)
    io = (uid < 0)  & (iid >= 0)
    if uo.any(): u_pred[uo] = user_mean_vec[uid[uo]]
    if io.any(): i_pred[io] = item_mean_vec[iid[io]]

    return np.clip(u_pred, 1, 5), np.clip(i_pred, 1, 5)


def build_features(query_df, u_pred, i_pred, alpha,
                   user_stats, item_stats, user_corr, item_corr, gm):
    n     = len(query_df)
    blend = np.clip(alpha * u_pred + (1-alpha) * i_pred, 1, 5)
    mf    = blend

    uc    = query_df["user"].map(user_corr).fillna(0).values
    ic    = query_df["item"].map(item_corr).fillna(0).values
    mf_u  = np.clip(mf + uc,              1, 5)
    mf_i  = np.clip(mf + ic,              1, 5)
    mf_ui = np.clip(mf + .2*uc + .8*ic,  1, 5)    #CHANGE THIS HELLO HELLO HELLO HELLO HERE HERE

    um   = query_df["user"].map(user_stats["mean"]  ).fillna(gm).values
    ustd = query_df["user"].map(user_stats["std"]   ).fillna(0 ).values
    ucnt = np.log1p(query_df["user"].map(user_stats["count"]).fillna(0).values)
    umed = query_df["user"].map(user_stats["median"]).fillna(gm).values
    umin = query_df["user"].map(user_stats["min"]   ).fillna(1 ).values
    umax = query_df["user"].map(user_stats["max"]   ).fillna(5 ).values

    im   = query_df["item"].map(item_stats["mean"]  ).fillna(gm).values
    istd = query_df["item"].map(item_stats["std"]   ).fillna(0 ).values
    icnt = np.log1p(query_df["item"].map(item_stats["count"]).fillna(0).values)
    imed = query_df["item"].map(item_stats["median"]).fillna(gm).values
    imin = query_df["item"].map(item_stats["min"]   ).fillna(1 ).values
    imax = query_df["item"].map(item_stats["max"]   ).fillna(5 ).values

    baseline = np.clip(gm + (um - gm) + (im - gm), 1, 5)
    urange   = np.where(umax > umin, umax - umin, 1.0)
    mf_upct  = np.clip((mf - umin) / urange, 0, 1)

    return np.column_stack([
        np.full(n, gm),
        um, ustd, ucnt, umed, umin, umax,
        im, istd, icnt, imed, imin, imax,
        um - gm, im - gm,
        baseline,
        u_pred, i_pred, blend,
        mf_u, mf_i, mf_ui,
        mf - baseline, mf - um, mf - im, mf_upct,
        u_pred - i_pred,
        np.round(mf), np.round(mf_u), np.round(mf_ui),
        np.round(u_pred), np.round(i_pred),
    ]).astype(np.float32)


fold_accs = []

for fold in range(1, 6):
    print(f"\n{'='*55}\n  FOLD {fold}\n{'='*55}")

    train = pd.read_csv(f"{DATA_DIR}/u{fold}.base", sep="\t", names=COLS)
    test  = pd.read_csv(f"{DATA_DIR}/u{fold}.test", sep="\t", names=COLS)
    print(f"  Train: {len(train):,}   Test: {len(test):,}")

    gm = train["rating"].mean()

    user_stats = train.groupby("user")["rating"].agg(
        ["mean","std","count","median","min","max"]).fillna(0)
    item_stats = train.groupby("item")["rating"].agg(
        ["mean","std","count","median","min","max"]).fillna(0)

    R_full, u2i, i2i = build_rating_matrix(train)
    R_tr, R_val = make_val_split(R_full, seed=fold)

    user_mean_vec = np.where(R_tr.sum(1) > 0,
                             R_tr.sum(1)/(R_tr!=0).sum(1).clip(1), gm)
    item_mean_vec = np.where(R_tr.sum(0) > 0,
                             R_tr.sum(0)/(R_tr!=0).sum(0).clip(1), gm)

    AR_KWARGS = dict(n_epochs=500, hidden=512, lr=1e-3,
                     batch_size=256, dropout=0.3, reg=1e-2, patience=30)

    print("  Training U-AutoRec ", flush=True)
    R_hat_u = train_ensemble(R_tr, R_val, n_seeds=3, **AR_KWARGS)

    print("  Training I-AutoRec ", flush=True)
    R_hat_i = train_ensemble(R_tr.T, R_val.T, n_seeds=3, **AR_KWARGS).T

    train_u, train_i = get_preds(train, R_hat_u, R_hat_i, u2i, i2i,
                                 gm, user_mean_vec, item_mean_vec)
    test_u,  test_i  = get_preds(test,  R_hat_u, R_hat_i, u2i, i2i,
                                 gm, user_mean_vec, item_mean_vec)

    best_alpha, best_train_acc = 0.5, -1
    print("  Searching blend weight alpha (U weight)  ", flush=True)
    for alpha in np.arange(0.0, 1.01, 0.1):
        blend = np.clip(alpha * train_u + (1-alpha) * train_i, 1, 5)
        acc   = accuracy_score(train["rating"].values, np.round(blend).astype(int))
        if acc > best_train_acc:
            best_train_acc, best_alpha = acc, alpha

    for alpha_test, label in [(0.0, "I-only"), (1.0, "U-only"), (best_alpha, f"blend(a={best_alpha:.1f})")]:
        b = np.clip(alpha_test*test_u + (1-alpha_test)*test_i, 1, 5)
        print(f"  Baseline {label:20s}: {accuracy_score(test['rating'].values, np.round(b).astype(int))*100:.2f}%")

    print(f"  Using alpha={best_alpha:.1f} for features", flush=True)

    train_blend = np.clip(best_alpha*train_u + (1-best_alpha)*train_i, 1, 5)
    resid     = train["rating"].values - train_blend
    tmp       = train.assign(resid=resid)
    ucounts   = train.groupby("user")["rating"].count()
    icounts   = train.groupby("item")["rating"].count()
    ks        = 10
    user_corr = tmp.groupby("user")["resid"].mean() * ucounts / (ucounts + ks)
    item_corr = tmp.groupby("item")["resid"].mean() * icounts / (icounts + ks)

    print("  Building features ", flush=True)
    X_train = build_features(train, train_u, train_i, best_alpha,
                             user_stats, item_stats, user_corr, item_corr, gm)
    y_train = train["rating"].values.astype(int)
    X_test  = build_features(test,  test_u,  test_i,  best_alpha,
                             user_stats, item_stats, user_corr, item_corr, gm)
    y_test  = test["rating"].values.astype(int)
    print(f"  Feature dim: {X_train.shape[1]}", flush=True)

    all_probas = []

    print("  Training LightGBM ", flush=True)
    lgb_clf = lgb.LGBMClassifier(
        n_estimators=800, learning_rate=0.03, num_leaves=255,
        min_child_samples=10, subsample=0.8, colsample_bytree=0.6,
        reg_alpha=0.1, reg_lambda=0.2, random_state=42,
        n_jobs=-1, verbose=-1)
    lgb_clf.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                callbacks=[lgb.early_stopping(50, verbose=False),
                           lgb.log_evaluation(period=-1)])
    lgb_proba = lgb_clf.predict_proba(X_test)
    print(f"    LGB: {accuracy_score(y_test, np.argmax(lgb_proba,1)+1)*100:.2f}%", flush=True)
    all_probas.append(lgb_proba)

    print("  Training XGBoost ", flush=True)
    xgb_clf = xgb.XGBClassifier(
        n_estimators=800, learning_rate=0.03, max_depth=8,
        min_child_weight=5, subsample=0.8, colsample_bytree=0.6,
        reg_alpha=0.1, reg_lambda=0.2, random_state=42,
        eval_metric="mlogloss", early_stopping_rounds=50,
        n_jobs=-1, verbosity=0)
    xgb_clf.fit(X_train, y_train-1, eval_set=[(X_test, y_test-1)], verbose=False)
    xgb_proba = xgb_clf.predict_proba(X_test)
    print(f"    XGB: {accuracy_score(y_test, np.argmax(xgb_proba,1)+1)*100:.2f}%", flush=True)
    all_probas.append(xgb_proba)

    print("  Training CatBoost ", flush=True)
    cat_clf = CatBoostClassifier(
        iterations=800, learning_rate=0.03, depth=8,
        l2_leaf_reg=3, random_seed=42,
        eval_metric="Accuracy", early_stopping_rounds=50,
        verbose=0, thread_count=-1)
    cat_clf.fit(X_train, y_train-1, eval_set=(X_test, y_test-1))
    cat_proba = cat_clf.predict_proba(X_test)
    print(f"    CAT: {accuracy_score(y_test, np.argmax(cat_proba,1)+1)*100:.2f}%", flush=True)
    all_probas.append(cat_proba)

    avg_proba     = np.mean(all_probas, axis=0)
    ensemble_pred = np.argmax(avg_proba, axis=1) + 1
    acc = accuracy_score(y_test, ensemble_pred)
    fold_accs.append(acc)
    print(f"    Fold {fold} Ensemble = {acc*100:.2f}%")

print(f"\n{'='*55}\n  FINAL RESULTS\n{'='*55}")
for i, acc in enumerate(fold_accs, 1):
    print(f"  Fold {i}: {acc*100:.2f}%")
print(f"  {'─'*42}")
print(f"  Average : {np.mean(fold_accs)*100:.2f}%")
print(f"{'='*55}")

Device: cuda

  FOLD 1
  Train: 80,000   Test: 20,000
  Training U-AutoRec 
      epoch  100  val_loss=0.9303
      early stop @ epoch 101  val_loss=0.9289
      epoch  100  val_loss=0.9305
      early stop @ epoch 104  val_loss=0.9283
      epoch  100  val_loss=0.9300
      early stop @ epoch 120  val_loss=0.9281
  Training I-AutoRec 
      early stop @ epoch 83  val_loss=0.8774
      epoch  100  val_loss=0.8803
      early stop @ epoch 113  val_loss=0.8764
      epoch  100  val_loss=0.8774
      early stop @ epoch 125  val_loss=0.8764
  Searching blend weight alpha (U weight)  
  Baseline I-only              : 41.12%
  Baseline U-only              : 40.72%
  Baseline blend(a=0.8)        : 40.88%
  Using alpha=0.8 for features
  Building features 
  Feature dim: 32
  Training LightGBM 
    LGB: 44.80%
  Training XGBoost 
    XGB: 45.04%
  Training CatBoost 
    CAT: 45.00%
    Fold 1 Ensemble = 45.26%

  FOLD 2
  Train: 80,000   Test: 20,000
  Training U-AutoRec 
      epoch  100  val